# Feature Store Validation

## Objective

Validate that `feature.loan_features_v1` is a production-ready analytical layer.

This notebook answers:

1. Was every loan carried into the feature store?
2. Was loan-level granularity preserved?
3. What features are available?
4. Are key engineered features populated correctly?
5. Are feature values economically reasonable?
6. Does the target variable behave as expected?
7. Does risk increase monotonically across LendingClub grades?


In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

DB_PATH = (
    Path.cwd().parent.parent
    / "data"
    / "warehouse"
    / "duckdb"
    / "lendingclub.duckdb"
)

print(DB_PATH)

conn = duckdb.connect(str(DB_PATH))


d:\project_lighthouse\projects\P0_Data_Platform\datasets\lendingclub\data\warehouse\duckdb\lendingclub.duckdb


## Portfolio Reconciliation

The feature layer should contain exactly the same number of loans as the clean layer because feature engineering is performed at loan level without aggregation or filtering.


In [2]:
conn.execute("""
select
    (select count(*) from clean.lendingclub_clean) as clean_rows,
    (select count(*) from feature.loan_features_v1) as feature_rows
""").fetchdf()


,clean_rows,feature_rows
0,2260668,2260668


### Finding

The clean layer and feature layer both contain 2,260,668 records.

Implication

No loans were lost, duplicated, or filtered during feature engineering. The feature store represents the complete LendingClub portfolio available in the clean layer.

Decision

Portfolio reconciliation passed and the feature layer can be treated as a complete downstream source.

## Feature Store Structure

Review the final schema before validating feature behaviour.


In [3]:
schema = conn.execute("""
describe feature.loan_features_v1
""").fetchdf()

print(f"Total Columns: {len(schema)}")

schema


Total Columns: 84


,column_name,column_type,null,key,default,extra
0,loan_id,BIGINT,YES,None,None,None
1,issue_date,TIMESTAMP,YES,None,None,None
2,origination_date,TIMESTAMP,YES,None,None,None
3,earliest_credit_date,TIMESTAMP,YES,None,None,None
4,issue_year,BIGINT,YES,None,None,None
5,issue_quarter,BIGINT,YES,None,None,None
6,issue_month_number,BIGINT,YES,None,None,None
7,issue_year_month,VARCHAR,YES,None,None,None
8,loan_age_months,BIGINT,YES,None,None,None
9,loan_amnt,DOUBLE,YES,None,None,None


### Finding

The feature store contains 84 columns.

The schema combines:
- Original LendingClub attributes
- Standardized date fields
- Credit-history metrics
- Ratio-based features
- Utilization measures
- Risk indicator flags
- Target variables

Implication

The dataset has successfully evolved from a cleaned operational dataset into a reusable analytical layer suitable for reporting, segmentation, monitoring, and future modelling.

Decision

The feature inventory is sufficiently broad for Mart Layer development.

## Sample of Engineered Features

Review a sample of the most important derived attributes.


In [4]:
engineered_features = [

    "issue_year",
    "issue_quarter",
    "issue_month_number",
    "issue_year_month",
    "loan_age_months",
    "credit_history_months",
    "credit_history_years",
    "fico_midpoint",
    "last_fico_midpoint",
    "fico_change",
    "loan_to_income_ratio",
    "installment_to_income_ratio",
    "revolving_balance_to_income",
    "total_balance_to_income",
    "high_utilization_flag",
    "utilization_band",
    "verified_income_flag",
    "homeowner_flag",
    "recent_inquiry_flag",
    "thin_file_flag",
    "derogatory_history_flag",
    "bankruptcy_flag",
    "mortgage_flag",
    "joint_application_flag",
    "high_dti_flag",
    "default_flag"
]

conn.execute(f"""
select
    {",".join(engineered_features)}
from feature.loan_features_v1
limit 20
""").fetchdf()


,issue_year,issue_quarter,issue_month_number,issue_year_month,loan_age_months,credit_history_months,credit_history_years,fico_midpoint,last_fico_midpoint,fico_change,loan_to_income_ratio,installment_to_income_ratio,revolving_balance_to_income,total_balance_to_income,high_utilization_flag,utilization_band,verified_income_flag,homeowner_flag,recent_inquiry_flag,thin_file_flag,derogatory_history_flag,bankruptcy_flag,mortgage_flag,joint_application_flag,high_dti_flag,default_flag
0,2015,4,12,2015-12,126,148,12.33,677.0,562.0,-115.0,0.065455,0.026843,0.050273,0.140836,0,MEDIUM,0,1,0,0,0,0,1,0,0,0
1,2015,4,12,2015-12,126,192,16.00,717.0,697.0,-20.0,0.380000,0.151436,0.330308,0.607308,0,LOW,0,1,1,0,1,0,1,0,0,0
2,2015,4,12,2015-12,126,184,15.33,697.0,702.0,5.0,0.317460,0.082411,0.124905,0.296762,0,HIGH,0,1,0,0,0,0,1,1,0,0
3,2015,4,12,2015-12,126,87,7.25,787.0,677.0,-110.0,0.318182,0.090535,0.070927,0.474782,0,LOW,1,1,0,0,0,0,1,0,0,0
4,2015,4,12,2015-12,126,210,17.50,697.0,702.0,5.0,0.099585,0.033312,0.209982,0.917028,0,HIGH,1,1,1,0,1,0,1,0,0,0
5,2015,4,12,2015-12,126,338,28.17,692.0,757.0,65.0,0.351471,0.143005,0.259471,0.376412,0,HIGH,1,0,0,0,0,0,0,0,0,0
6,2015,4,12,2015-12,126,306,25.50,682.0,652.0,-30.0,0.111111,0.042505,0.485161,0.648678,1,VERY_HIGH,0,1,0,0,0,0,1,0,0,0
7,2015,4,12,2015-12,126,202,16.83,707.0,672.0,-35.0,0.235294,0.089119,0.009718,0.328671,0,LOW,0,1,0,0,1,0,1,0,0,0
8,2015,4,12,2015-12,126,164,13.67,687.0,717.0,30.0,0.117647,0.043264,0.123106,0.328906,0,MEDIUM,0,0,0,0,1,1,1,0,0,0
9,2015,4,12,2015-12,126,253,21.08,702.0,677.0,-25.0,0.190476,0.075354,0.167476,2.709095,0,MEDIUM,0,1,0,0,0,0,1,0,1,0


### Finding

The sampled records confirm that all major engineered feature families are being populated correctly.

Observed features include:
- Temporal variables (year, quarter, month, loan age)
- Credit-history measures
- FICO-based variables
- Income and leverage ratios
- Utilization measures
- Binary risk indicators
- Default target flag

Implication

Feature engineering logic is functioning as intended and producing values that are economically plausible.

Decision

No additional validation concerns were identified from the feature sample review.

## Loan-Level Integrity Check

The feature store must preserve one row per loan.


In [5]:
conn.execute("""
select
    count(*) as rows,
    count(distinct loan_id) as distinct_loans,
    count(*) - count(distinct loan_id) as duplicate_loans
from feature.loan_features_v1
""").fetchdf()


,rows,distinct_loans,duplicate_loans
0,2260668,2260668,0


### Finding

The feature store contains 2,260,668 records and 2,260,668 distinct loan IDs.

Duplicate loans identified: 0

Implication

Loan-level granularity has been preserved throughout the transformation process.

Decision

The feature store is suitable for aggregation, reporting, and downstream mart construction.

## Key Feature Completeness Audit

Audit only the most important engineered variables instead of scanning all 2.26 million records into pandas.


In [6]:
conn.execute("""
select
    sum(case when issue_year is null then 1 else 0 end) as issue_year_nulls,
    sum(case when fico_midpoint is null then 1 else 0 end) as fico_nulls,
    sum(case when loan_to_income_ratio is null then 1 else 0 end) as lti_nulls,
    sum(case when credit_history_years is null then 1 else 0 end) as credit_history_nulls,
    sum(case when utilization_band is null then 1 else 0 end) as utilization_band_nulls,
    sum(case when default_flag is null then 1 else 0 end) as default_nulls
from feature.loan_features_v1
""").fetchdf().T


,0
issue_year_nulls,0.0
fico_nulls,0.0
lti_nulls,1671.0
credit_history_nulls,29.0
utilization_band_nulls,0.0
default_nulls,0.0


### Finding

Feature completeness is extremely strong.

Null counts observed:

| Feature | Nulls |
|----------|----------:|
| issue_year | 0 |
| fico_midpoint | 0 |
| utilization_band | 0 |
| default_flag | 0 |
| credit_history_years | 29 |
| loan_to_income_ratio | 1,671 |

Relative to a population of 2.26 million loans, missingness is negligible.

Implication

The feature engineering process successfully populated all critical analytical variables.

Decision

No remediation is required before downstream mart development.

## Feature Range Validation

Validate that engineered variables fall within economically reasonable ranges.


In [7]:
conn.execute("""
select

    min(fico_midpoint) as min_fico,
    quantile_cont(fico_midpoint,0.25) as p25_fico,
    quantile_cont(fico_midpoint,0.50) as median_fico,
    quantile_cont(fico_midpoint,0.75) as p75_fico,
    max(fico_midpoint) as max_fico,

    min(loan_to_income_ratio) as min_lti,
    quantile_cont(loan_to_income_ratio,0.50) as median_lti,
    max(loan_to_income_ratio) as max_lti,

    min(credit_history_years) as min_credit_years,
    avg(credit_history_years) as avg_credit_years,
    max(credit_history_years) as max_credit_years,

    min(loan_age_months) as min_loan_age,
    avg(loan_age_months) as avg_loan_age,
    max(loan_age_months) as max_loan_age

from feature.loan_features_v1
""").fetchdf()


,min_fico,p25_fico,median_fico,p75_fico,max_fico,min_lti,median_lti,max_lti,min_credit_years,avg_credit_years,max_credit_years,min_loan_age,avg_loan_age,max_loan_age
0,612.0,677.0,692.0,717.0,847.5,0.000161,0.2,40000.0,0.5,16.394675,83.25,90,120.954388,228


### Finding

The feature ranges appear economically reasonable.

Key observations:

- FICO scores range from 612 to 848
- Median FICO score is 692
- Average credit history length is approximately 16.4 years
- Maximum credit history exceeds 83 years
- Median loan-to-income ratio is approximately 0.20

One notable observation is the presence of extreme leverage outliers, with loan-to-income ratios reaching 40,000.

Implication

The feature calculations appear valid. Extreme ratio values are likely driven by very small reported incomes rather than transformation defects.

Decision

The feature layer passes range validation. Future modelling projects may cap or winsorize ratio outliers if required.

## Portfolio Default Rate

Validate the target variable and measure overall portfolio risk.


In [8]:
conn.execute("""
select
    default_flag,
    count(*) as loans,
    round(
        100.0 * count(*) /
        sum(count(*)) over (),
        2
    ) as pct
from feature.loan_features_v1
group by default_flag
order by default_flag
""").fetchdf()


,default_flag,loans,pct
0,0,1969841,87.14
1,1,290827,12.86


### Finding

The portfolio contains:

- 1,969,841 non-default loans (87.14%)
- 290,827 default loans (12.86%)

Implication

The default flag has been constructed successfully and produces a realistic portfolio bad rate for a large unsecured consumer lending dataset.

Decision

The target variable is suitable for portfolio analytics and future modelling projects.

## Risk Ordering Validation

A fundamental sanity check: worse grades should exhibit higher default rates.


In [9]:
conn.execute("""
select

    grade,

    count(*) as loans,

    round(
        100.0 * count(*) /
        sum(count(*)) over (),
        2
    ) as portfolio_share_pct,

    sum(default_flag) as bad_loans,

    round(
        100.0 * avg(default_flag),
        2
    ) as bad_rate_pct

from feature.loan_features_v1

group by grade

order by grade
""").fetchdf()


,grade,loans,portfolio_share_pct,bad_loans,bad_rate_pct
0,A,433027,19.15,15536.0,3.59
1,B,663557,29.35,57449.0,8.66
2,C,650053,28.75,93359.0,14.36
3,D,324424,14.35,66025.0,20.35
4,E,135639,6.00,38365.0,28.28
5,F,41800,1.85,15225.0,36.42
6,G,12168,0.54,4868.0,40.01


### Finding

Default rates increase consistently across LendingClub grades:

| Grade | Bad Rate |
|---------|---------:|
| A | 3.59% |
| B | 8.66% |
| C | 14.36% |
| D | ~20% |
| E | ~28% |
| F | ~36% |
| G | ~40% |

The relationship is monotonic, meaning risk increases steadily as credit quality deteriorates.

Additional observation:

Grades B and C represent more than 58% of the entire portfolio, making them the most commercially important segments.

Implication

This is one of the strongest validation checks available because it simultaneously validates both target construction and feature quality.

Decision

Risk-ordering validation passed.

# Validation Summary

## Validation Result

Status: PASSED

All core feature-store validation checks were successfully completed.

| Validation Area | Result |
|----------|----------|
| Portfolio Reconciliation | Passed |
| Schema Validation | Passed |
| Feature Population Review | Passed |
| Loan Uniqueness Validation | Passed |
| Feature Completeness Validation | Passed |
| Feature Range Validation | Passed |
| Default Target Validation | Passed |
| Grade Risk Ordering Validation | Passed |

## Key Portfolio Findings

- Portfolio size: 2.26 million loans
- Feature count: 84 columns
- Portfolio default rate: 12.86%
- Median FICO score: 692
- Average credit history: 16.4 years
- No duplicate loans identified
- Critical engineered features exhibit near-zero missingness
- Credit grades display the expected monotonic risk ordering

## Conclusion

`feature.loan_features_v1` is a production-ready analytical layer.

The feature store preserves full portfolio coverage, maintains loan-level granularity, contains validated engineered features, and produces a realistic target variable suitable for downstream analytical consumption.

## Next Step

Proceed to Mart Layer development beginning with:

**M001 – Portfolio Summary Mart**